In [44]:
import open3d as o3d
import numpy as np
from sklearn.decomposition import PCA

pcd = o3d.io.read_point_cloud(
    # "/home/sha/Documents/mesh/MeshroomCache/ConvertSfMFormat/ebd642612b8b456855af6468c5b3ed3e2a943192/sfm.ply"
    # "/home/sha/Documents/work/robohand/calib/for_3d/MeshroomCache/ConvertSfMFormat/05d2c15ea6d9c1b6451f1c63a3ddab3ba05d0977/sfm.ply",
    # "cloud_000.ply"
    "/home/sha/Documents/work/robohand_v2/colmap/fused.ply"
)
print(f"Points: {len(pcd.points)}")

o3d.visualization.draw_geometries([pcd])

Points: 408427


In [45]:
pcd = pcd.voxel_down_sample(voxel_size=0.005)
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=1.0)

points = np.asarray(pcd.points)
x_values = points[:, 0]

z_min, z_max = np.percentile(x_values, [5, 85])
print(f"{z_min:.3f} - {z_max:.3f}")

mask = (points[:, 2] > z_min) & (points[:, 2] < z_max)
child_pcd = pcd.select_by_index(np.where(mask)[0])

-12.203 - 0.713


In [46]:
labels = np.array(
    child_pcd.cluster_dbscan(eps=0.1, min_points=10, print_progress=False)
)
max_label = labels.max()
print(f"{max_label + 1} clusters")

largest_cluster = np.argmax(np.bincount(labels[labels >= 0]))
final_pcd = child_pcd.select_by_index(np.where(labels == largest_cluster)[0])

4 clusters


In [47]:
o3d.visualization.draw_geometries([pcd, child_pcd])
o3d.visualization.draw_geometries([final_pcd])

In [21]:
# plane_model, inliers = final_pcd.segment_plane(
#     distance_threshold=0.01, ransac_n=8, num_iterations=100
# )
# table = final_pcd.select_by_index(inliers)
# object_pcd = final_pcd.select_by_index(inliers, invert=True)
# table.paint_uniform_color([0.8, 0.8, 0.8])
# object_pcd.paint_uniform_color([1, 0, 0])
# o3d.visualization.draw_geometries([table, object_pcd])

In [42]:
object_pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=10, std_ratio=2.0)

labels = np.array(
    object_pcd.cluster_dbscan(eps=0.01, min_points=5, print_progress=False)
)
max_label = labels.max()
print(f"{max_label + 1} clusters")

largest_cluster_idx = np.argmax(np.bincount(labels[labels >= 0]))
final_pcd = object_pcd.select_by_index(np.where(labels == largest_cluster_idx)[0])
final_pcd.transform(np.array([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]))
o3d.visualization.draw_geometries([final_pcd])

1232 clusters


In [7]:
# o3d.io.write_point_cloud("/home/sha/Documents/mesh/final_point_cloud.ply", final_pcd)

True

In [ ]:
final_pcd.estimate_normals()

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    final_pcd, depth=9
)

vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)

o3d.io.write_triangle_mesh("output_model.obj", mesh)

radius = 0.5
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    final_pcd, o3d.utility.DoubleVector([radius, radius * 1.3])
)

o3d.io.write_triangle_mesh("output_ball_pivot.obj", mesh)

In [22]:
mesh = o3d.io.read_triangle_mesh("output_model.obj")
o3d.visualization.draw_geometries([mesh])

In [23]:
mesh = o3d.io.read_triangle_mesh("output_ball_pivot.obj")
o3d.visualization.draw_geometries([mesh])